In [1]:
from ingest import load_faq_data
documents = load_faq_data()

documents_llm = []

for doc in documents:
    if doc["course"] == "llm-zoomcamp":
        documents_llm.append(doc)

In [2]:
from evaluation_utils import llm_structured_retry

In [3]:
from pydantic import BaseModel

class Questions(BaseModel):
    questions: list[str]

In [4]:
import json

In [5]:
data_gen_instructions = """
You emulate a student who's taking our course.
Formulate 5 questions this student might ask based on a FAQ record. The record
should contain the answer to the questions, and the questions should be complete and not too short.
If possible, use as fewer words as possible from the record.

The output should resemble how people ask questions
on the internet. Not too formal, not too short, not too long.
""".strip()

In [6]:
from openai import OpenAI
from dotenv import load_dotenv
import os

load_dotenv()

token = os.getenv("GEMINI_API_KEY")
endpoint = "https://generativelanguage.googleapis.com/v1beta/openai/"
model='gemini-2.5-flash'

openai_client = OpenAI(
    base_url=endpoint,
    api_key=token,
)

In [7]:
def generate_ground_truth(doc):
    user_prompt = json.dumps(doc)

    out, usage = llm_structured_retry(
        openai_client,
        data_gen_instructions,
        user_prompt,
        Questions
    )

    results = []

    for q in out.questions:
        results.append({
            "question": q,
            "document": doc["id"]
        })

    return results, usage

In [8]:
from concurrent.futures import ThreadPoolExecutor
from evaluation_utils import map_progress

In [ ]:
with ThreadPoolExecutor(max_workers=3) as pool:
    results = map_progress(pool, documents, generate_ground_truth)

  0%|          | 0/1375 [00:00<?, ?it/s]

In [ ]:
ground_truth = []
usages = []

for records, usage in results:
    ground_truth.extend(records)
    usages.append(usage)

len(ground_truth)